# FluidX3D Force Validation Lab

Use this notebook to run, label, collect, and plot force-validation experiments for NACA, Ahmed, and skijumper cases.

The C++ executable is still the source of truth for simulations. This notebook only organizes runs and reads CSV output from `bin/export/force_validation/`.

## First-Time Setup

From the repo root in a VS Code terminal:

```powershell
python -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install -r py\requirements.txt
```

If PowerShell blocks `Activate.ps1`, run this once:

```powershell
Set-ExecutionPolicy -Scope CurrentUser -ExecutionPolicy RemoteSigned
```

Then open this notebook and select the `.venv` kernel. You only need to do this once per machine.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "py"))

import force_validation as fv

ROOT, fv.local_executable(ROOT), fv.export_dir(ROOT)

## Define a Local Run

Keep `RUN_NOW = False` until the command looks right. Set it to `True` to actually launch the executable.

In [ ]:
RUN_NOW = False

local_spec = fv.RunSpec(
    case="all",      # all, naca, ahmed, skijumper
    memory_mb=1000,
    gpu_ids=("0",),
    label="local-debug-skijumper-512mb",
    location="local-5080-16gb",
    notes="First testruns 1GB",
)

result = fv.run_local(local_spec, ROOT, dry_run=not RUN_NOW)
result

If `RUN_NOW = True`, inspect stdout/stderr here. The helper writes an entry to `py/run_log.csv` after the process exits.

In [ ]:
if RUN_NOW:
    print("returncode:", result.returncode)
    print("--- stdout ---")
    print(result.stdout[-4000:])
    print("--- stderr ---")
    print(result.stderr[-4000:])

## Remote Run Scaffold

This is intentionally dry-run for now. Once the Linux server details are known, fill in `host`, `user`, `repo_path`, and `ssh_port`.

Expected server setup later: repo cloned/built on the server, STLs present, and SSH access from this PC.

In [ ]:
REMOTE_RUN_NOW = False
REMOTE_COPY_NOW = False

remote = fv.RemoteConfig(
    host="your.server.host",
    user="your_user",
    repo_path="/home/your_user/fxms",
    ssh_port=22,
    executable="bin/FluidX3D",
)

remote_spec = fv.RunSpec(
    case="all",
    memory_mb=40000,
    gpu_ids=("0",),
    label="remote-all-40gb",
    location="linux-server-56gb-vram",
    notes="High-resolution remote run placeholder",
)

remote_cmd = fv.run_remote(remote_spec, remote, dry_run=not REMOTE_RUN_NOW)
copy_cmd = fv.copy_remote_csvs(remote, destination=fv.export_dir(ROOT), dry_run=not REMOTE_COPY_NOW)

remote_cmd, copy_cmd

## Load CSV Output

This reads all CSV files currently in `bin/export/force_validation/`.

In [ ]:
df = fv.load_force_csvs(ROOT)
log = fv.load_run_log(ROOT)

print(f"CSV rows: {len(df)}")
print(f"Logged runs: {len(log)}")
df.head()

In [ ]:
fv.latest_samples(df)

In [ ]:
fv.summarize_runs(df)

## Plot Force Convergence

In [ ]:
import matplotlib.pyplot as plt

if df.empty:
    print("No CSV data yet. Run an experiment first, then reload this cell.")
else:
    for case, case_df in df.groupby("case"):
        fig, ax = plt.subplots(figsize=(10, 5))
        for source_file, run_df in case_df.groupby("source_file"):
            ax.plot(run_df["step"], run_df["Fy_drag_N"], marker="o", label=f"drag {source_file}")
            ax.plot(run_df["step"], run_df["Fz_lift_N"], marker="x", label=f"lift {source_file}")
        ax.set_title(f"{case}: force convergence")
        ax.set_xlabel("LBM step")
        ax.set_ylabel("Force [N]")
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.show()

## Plot Coefficients

In [ ]:
if df.empty:
    print("No CSV data yet. Run an experiment first, then reload this cell.")
else:
    for case, case_df in df.groupby("case"):
        fig, ax = plt.subplots(figsize=(10, 5))
        for source_file, run_df in case_df.groupby("source_file"):
            ax.plot(run_df["step"], run_df["Cd"], marker="o", label=f"Cd {source_file}")
            ax.plot(run_df["step"], run_df["Cl"], marker="x", label=f"Cl {source_file}")
        ax.set_title(f"{case}: coefficient convergence")
        ax.set_xlabel("LBM step")
        ax.set_ylabel("Coefficient")
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.show()

## Notes for Interpreting Runs

- For skijumper, raw `Fy_drag_N` and `Fz_lift_N` are the primary quantities.
- Skijumper coefficients use dummy-bound box area, so treat them as run-to-run comparators rather than final wind-tunnel coefficients.
- Ahmed 25 deg target band: `Cd = 0.285-0.298`.
- Skijumper target bands: drag `40-45 N`, lift `28-32 N`.
- NACA 0 deg target band: `Cd = 0.006-0.008` as a sanity check.
- For memory sweeps, compare the last row of each CSV first, then inspect convergence plots.

## Average Lift, Drag, and L/D Ratio

This averages all sampled rows in each CSV run. Use it for quick run-to-run comparisons after checking the convergence plots.

In [ ]:
import pandas as pd

if df.empty:
    print("No CSV data yet. Run an experiment first, then reload this cell.")
else:
    averages = (
        df.groupby(["case", "memory_mb", "source_file"], as_index=False)
        .agg(
            samples=("step", "count"),
            drag_N_mean=("Fy_drag_N", "mean"),
            drag_N_std=("Fy_drag_N", "std"),
            lift_N_mean=("Fz_lift_N", "mean"),
            lift_N_std=("Fz_lift_N", "std"),
        )
    )
    averages["L_to_D_mean"] = averages["lift_N_mean"] / averages["drag_N_mean"].replace(0, pd.NA)
    display(averages.sort_values(["case", "memory_mb", "source_file"]))

In [ ]:
if df.empty:
    print("No CSV data yet. Run an experiment first, then reload this cell.")
else:
    latest = fv.latest_samples(df).copy()
    checks = []
    for _, row in latest.iterrows():
        case = row["case"]
        if case == "ahmed":
            checks.append({**row.to_dict(), "metric": "Cd", "value": row["Cd"], "target_min": 0.285, "target_max": 0.298})
        elif case == "skijumper":
            checks.append({**row.to_dict(), "metric": "drag_N", "value": row["Fy_drag_N"], "target_min": 40.0, "target_max": 45.0})
            checks.append({**row.to_dict(), "metric": "lift_N", "value": row["Fz_lift_N"], "target_min": 28.0, "target_max": 32.0})
        elif case == "naca":
            checks.append({**row.to_dict(), "metric": "Cd", "value": row["Cd"], "target_min": 0.006, "target_max": 0.008})
    target_df = pd.DataFrame(checks)
    if target_df.empty:
        print("No target-band checks apply to the loaded cases.")
    else:
        target_df["in_band"] = target_df["value"].between(target_df["target_min"], target_df["target_max"])
        display(target_df[["case", "memory_mb", "source_file", "metric", "value", "target_min", "target_max", "in_band"]])

In [ ]:
if df.empty:
    print("No CSV data yet. Run an experiment first, then reload this cell.")
else:
    plot_df = averages.copy()
    plot_df["run"] = plot_df["case"] + " " + plot_df["memory_mb"].astype(str) + "MB"
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    plot_df.plot.bar(x="run", y="drag_N_mean", ax=axes[0], legend=False, title="Average Drag [N]")
    plot_df.plot.bar(x="run", y="lift_N_mean", ax=axes[1], legend=False, title="Average Lift [N]")
    plot_df.plot.bar(x="run", y="L_to_D_mean", ax=axes[2], legend=False, title="Average L/D")
    for ax in axes:
        ax.grid(True, axis="y", alpha=0.3)
        ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()